In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [14]:
#1. 位置编码
class positional_encoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -math.log(10000.0) / d_model)
        
        pe[:, 0::2] = torch.sin(position * div_term)#偶数位置   
        pe[:, 1::2] = torch.cos(position * div_term)#奇数位置
        pe = pe.unsqueeze(0)#增加一个维度 (1, max_len, d_model)

        self.register_buffer('pe', pe)#pe存放到寄存器

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]#(1, max_len, d_model) “1” 广播
        return x


        
        

In [15]:
#2. 多头自注意力机制
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0 #d_model 必须能被 num_heads 整除

        self.d_model = d_model
        self.num_heads = num_heads

        self.head_dim = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

        self.out = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        #x.shape = (bs, seq_len, d_model) → (bs, num_heads, seq_len, head_dim)
        batch_size, seq_len, _ = x.size()
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 2)#(bs, num_heads, seq_len, head_dim)
        return x



    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)
        return output


    def merge_heads(self, x):
        #x.shape = (bs, num_heads, seq_len, head_dim) → (bs, seq_len, d_model)
        batch_size, _, seq_len, _ = x.size()
        x = x.transpose(1, 2)#(bs, seq_len, num_heads, head_dim)
        x = x.reshape(batch_size, seq_len, self.d_model)
        return x


    def forward(self, Q, K, V, mask=None):
        #Q,K,V.shape = (bs, seq_len, d_model) 
        Q = self.split_heads(self.Wq(Q))
        K = self.split_heads(self.Wk(K))
        V = self.split_heads(self.Wv(V))

        #注意力计算
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)#(bs, num_heads, seq_len, head_dim)

        #合并多头
        output = self.merge_heads(attn_output)
        output = self.out(output)
        return output


In [16]:
#3. 前馈神经网络
class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.linear2(self.dropout((F.relu(self.linear1(x)))))
        return x


In [5]:
#4. Encoder层

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, hidden_dim, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, hidden_dim, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        #多头自注意力机制+残差+归一化
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        #前馈神经网络 + 残差 + 归一化
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x



In [20]:
#5. Decoder层
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, hidden_dim, dropout=0.1):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, hidden_dim, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        #Mask 自注意力机制
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))

        #cross注意力
        cross_output = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_output))

        #前馈网络
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x



In [21]:
#6. Transformer模型
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, 
                num_layers=6, hidden_dim=2048, max_seq_len=5000, dropout=0.1 ):
        super().__init__()

        #Embedding层
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoding = positional_encoding(d_model, max_seq_len)

        #Encoder 层
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, hidden_dim, dropout) for _ in range(num_layers)
        ])

        #Decoder 层
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, hidden_dim, dropout) for _ in range(num_layers)
        ])

        #输出层
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model


    def generate_mask(self, src, tgt):
        #源序列的 padding mask
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)

        #目标序列的 padding mask
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)

        #目标序列casual mask（防止看到未来的信息）
        seq_len = tgt.size(1)
        nopeak_mask = torch.tril(torch.ones(1, seq_len, seq_len)).bool()
        tgt_mask = tgt_mask & nopeak_mask

        return src_mask, tgt_mask


    def forward(self, src, tgt):
        #src:(batch, seq_len), tgt:(batch, seq_len)
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        
        #Encoder
        src_emb = self.dropout(self.pos_encoding(self.src_embedding(src)))
        enc_output = src_emb
        for layer in self.encoder_layers:
            enc_output = layer(enc_output, src_mask)
            
        #Decoder
        tgt_emb = self.dropout(self.pos_encoding(self.tgt_embedding(tgt)))
        dec_output = tgt_emb
        for layer in self.decoder_layers:
            dec_output = layer(dec_output, enc_output, src_mask, tgt_mask)
        
        #输出层
        output = self.fc_out(dec_output)
        
        return output
        
        


In [22]:
#7. 测试模型
# 模型参数
src_vocab_size=5000
tgt_vocab_size=5000
d_model=512
num_heads=8
num_layers=6
hidden_dim=2048
dropout=0.1

# 创建模型
model=Transformer(src_vocab_size,tgt_vocab_size,d_model,num_heads,num_layers,hidden_dim,dropout=dropout)
#生成随机输入
batch_size=2
src_seq_len=10
tgt_seq_len=8

src=torch.randint(1,src_vocab_size,(batch_size,src_seq_len))
tgt=torch.randint(1,tgt_vocab_size,(batch_size,tgt_seq_len))

# 前向传播
output = model(src,tgt)

print(f"源序列形状: {src.shape}")
print(f"目标序列形状: {tgt.shape}")
print(f"输出形状: {output.shape}")
print(f"\n模型总参数量: {sum(p.numel()for p in model.parameters()):,}")
print("\n✅模型运行成功！")

源序列形状: torch.Size([2, 10])
目标序列形状: torch.Size([2, 8])
输出形状: torch.Size([2, 8, 5000])

模型总参数量: 51,823,496

✅模型运行成功！


In [23]:
#8. 模型结构可视化
print(model)

Transformer(
  (src_embedding): Embedding(5000, 512)
  (tgt_embedding): Embedding(5000, 512)
  (pos_encoding): positional_encoding()
  (encoder_layers): ModuleList(
    (0-5): 6 x EncoderLayer(
      (self_attn): MultiHeadAttention(
        (Wq): Linear(in_features=512, out_features=512, bias=True)
        (Wk): Linear(in_features=512, out_features=512, bias=True)
        (Wv): Linear(in_features=512, out_features=512, bias=True)
        (out): Linear(in_features=512, out_features=512, bias=True)
      )
      (feed_forward): FeedForward(
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (decoder_layers): ModuleList(
    (0-5): 6 x DecoderLay